<a href="https://colab.research.google.com/github/THEOU-JEFFINE/gui-perturb/blob/main/experiments_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. fresh Colab with GPU runtime (Runtime → Change runtime type → T4)
!git clone https://github.com/THEOU-JEFFINE/gui-perturb.git
%cd gui-perturb
!pip install -q transformers accelerate qwen-vl-utils datasets

Cloning into 'gui-perturb'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 22 (delta 5), reused 15 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 921.09 KiB | 7.87 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/gui-perturb
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 58.3 MB/s eta 0:00:00


In [ ]:
from google.colab import files
files.upload()              # pick perturbed.zip
!unzip -q perturbed.zip -d /content

Saving perturbed.zip to perturbed.zip


In [ ]:
import os
print(os.listdir("/content/perturbed"))   # expect: clean, overlay, injection, theme, resolution, manifest.csv

['resolution', 'clean', 'overlay', 'theme', 'injection', 'manifest.csv']


In [ ]:
import sys; sys.path.insert(0, "/content/gui-perturb")
from gui_perturb.evaluate import ShowUIModel, run_evaluation, load_manifest
model = ShowUIModel()
model.model = model.model.to("cuda")
print("model on:", next(model.model.parameters()).device)   # want cuda:0

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model on: cuda:0


In [ ]:
from datasets import load_dataset
from gui_perturb import dataset as D
ds = load_dataset("HongxinLi/ScreenSpot_v2", split="test")
D.BBOX_FMT="xyxy"; D.BBOX_NORMALIZED=True
print(D.generate_perturbed_set(ds, out_dir="/content/perturbed", n=100, seed=0))

README.md:   0%|          | 0.00/731 [00:00<?, ?B/s]

data/test-00000-of-00002.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/295M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1272 [00:00<?, ? examples/s]

{'requested': 100, 'written': 100, 'bad_boxes': 0, 'splits': ['clean', 'overlay', 'injection', 'theme', 'resolution'], 'manifest': '/content/perturbed/manifest.csv'}


In [ ]:
import json
base = run_evaluation("/content/perturbed","/content/perturbed/manifest.csv",
    splits=["clean","overlay","injection","theme","resolution"], mitigate_splits=[], limit=None)
print(json.dumps(base, indent=2))
json.dump(base, open("/content/base_results.json","w"), indent=2)
from google.colab import files; files.download("/content/base_results.json")

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Running split: clean
  [clean] 25/200
  [clean] 125/200
  [clean] 150/200
  [clean] 175/200
  [clean] 200/200
Running split: overlay
  [overlay] 25/200
  [overlay] 125/200
  [overlay] 150/200
  [overlay] 175/200
  [overlay] 200/200
Running split: injection
  [injection] 25/200
  [injection] 125/200
  [injection] 150/200
  [injection] 175/200
  [injection] 200/200
Running split: theme
  [theme] 25/200
  [theme] 125/200
  [theme] 150/200
  [theme] 175/200
  [theme] 200/200
Running split: resolution
  [resolution] 25/200
  [resolution] 125/200
  [resolution] 150/200
  [resolution] 175/200
  [resolution] 200/200
{
  "clean": {
    "accuracy": 0.76,
    "correct": 76,
    "total": 100,
    "by_type": {
      "icon": 0.6667,
      "text": 0.8462
    }
  },
  "overlay": {
    "accuracy": 0.73,
    "correct": 73,
    "total": 100,
    "by_type": {
      "icon": 0.6875,
      "text": 0.7692
    }
  },
  "injection": {
    "accuracy": 0.74,
    "correct": 74,
    "total": 100,
    "by_type": {
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
mit = run_evaluation("/content/perturbed","/content/perturbed/manifest.csv",
    splits=[], mitigate_splits=["overlay","injection","theme","resolution"], limit=None)
print(json.dumps(mit, indent=2))
json.dump(mit, open("/content/mit_results.json","w"), indent=2)
files.download("/content/mit_results.json")

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Running split (mitigation): overlay
  [overlay+mitigation] 25/200
  [overlay+mitigation] 125/200
  [overlay+mitigation] 150/200
  [overlay+mitigation] 175/200
  [overlay+mitigation] 200/200
Running split (mitigation): injection
  [injection+mitigation] 25/200
  [injection+mitigation] 125/200
  [injection+mitigation] 150/200
  [injection+mitigation] 175/200
  [injection+mitigation] 200/200
Running split (mitigation): theme
  [theme+mitigation] 25/200
  [theme+mitigation] 125/200
  [theme+mitigation] 150/200
  [theme+mitigation] 175/200
  [theme+mitigation] 200/200
Running split (mitigation): resolution
  [resolution+mitigation] 25/200
  [resolution+mitigation] 125/200
  [resolution+mitigation] 150/200
  [resolution+mitigation] 175/200
  [resolution+mitigation] 200/200
{
  "overlay+mitigation": {
    "accuracy": 0.69,
    "correct": 69,
    "total": 100,
    "by_type": {
      "icon": 0.625,
      "text": 0.75
    }
  },
  "injection+mitigation": {
    "accuracy": 0.43,
    "correct": 43

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>